In [321]:
import joblib
from tensorflow import keras

# feature columns
price_cols = [
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "MA10",
    "MA20",
    "EMA10",
    "RSI",
    "MACD",
    "MACD_Signal",
    "MACD_Hist"
]

sentiment_cols = [
    "sentiment_mean",
    "sentiment_3d",
    "sentiment_7d",
    "sentiment_14d"
]

label_cols = [
    "label"
]

# load model
model = keras.models.load_model(
    "E:/3rd_sem_project/stock_price_predictor_models/v1/6th_sem_final_cnn_bilstm_sentiment_rsi_macd_multidataset/stock_model.keras"
)

# load scaler dictionary
feature_scalers = joblib.load(
    "E:/3rd_sem_project/stock_price_predictor_models/v1/6th_sem_final_cnn_bilstm_sentiment_rsi_macd_multidataset/feature_scalers.pkl"
)

close_scalers = joblib.load(
    "E:/3rd_sem_project/stock_price_predictor_models/v1/6th_sem_final_cnn_bilstm_sentiment_rsi_macd_multidataset/close_scalers.pkl"
)

# choose ticker
label = "MSFT"

label_maps = {
    "MSFT":1.0,
    "AAPL":2.0,
    "AMZN":3.0,
    "TSLA":4.0,
    "NVDA":5.0
}

feature_scaler = feature_scalers[label_maps[label]]
close_scaler = close_scalers[label_maps[label]]

In [322]:
print(feature_scalers)

{1.0: MinMaxScaler(), 2.0: MinMaxScaler(), 3.0: MinMaxScaler(), 4.0: MinMaxScaler(), 5.0: MinMaxScaler()}


In [323]:
for l, scaler in feature_scalers.items():
    print(f"\n===== Scaler for label {l} =====")
    print("Number of features:", scaler.n_features_in_)
    print("Min values:", scaler.data_min_)
    print("Max values:", scaler.data_max_)


===== Scaler for label 1.0 =====
Number of features: 12
Min values: [ 1.11131868e+01  1.14579516e+01  1.09077939e+01  1.11498642e+01
  5.85590000e+06  1.17000206e+01  1.20653250e+01  1.17549468e+01
  6.80574198e+00 -1.88481670e+01 -1.76814259e+01 -6.19242109e+00]
Max values: [5.39825256e+02 5.52242002e+02 5.38530652e+02 5.52023241e+02
 3.19317900e+08 5.23633557e+02 5.19582895e+02 5.25596853e+02
 9.91116400e+01 1.85959497e+01 1.67023625e+01 7.39455725e+00]

===== Scaler for label 2.0 =====
Number of features: 12
Min values: [ 2.34310722e+00  2.45696647e+00  2.34310722e+00  2.37876267e+00
  1.79106000e+07  2.56123765e+00  2.61022724e+00  2.54379626e+00
  1.13054746e-01 -1.08943993e+02 -9.52607251e+01 -5.03457019e+01]
Max values: [2.85922455e+02 2.88350192e+02 2.83035157e+02 2.85932471e+02
 3.37296960e+09 3.82267802e+02 3.83426900e+02 3.44025195e+02
 9.74875519e+01 8.88666927e+00 8.25634031e+00 1.40854119e+01]

===== Scaler for label 3.0 =====
Number of features: 12
Min values: [ 1.75150

In [324]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert",
    device=-1
)

label_map = {
    "positive": 1,
    "negative": -1,
    "neutral": 0
}

Loading weights: 100%|███████████████████████████| 201/201 [00:00<00:00, 25111.10it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [325]:
import requests
import pandas as pd
from datetime import datetime, timedelta
from dotenv import load_dotenv
import os

load_dotenv()

API_KEY = os.getenv("NEWS_API")

def fetch_recent_news(days=16):
    end = datetime.today()
    start = end - timedelta(days=days)

    url = "https://newsapi.org/v2/everything"

    params = {
        "q": "Microsoft OR MSFT",
        "from": start.strftime("%Y-%m-%d"),
        "to": end.strftime("%Y-%m-%d"),
        "language": "en",
        "apiKey": os.getenv("NEWS_API")
    }

    r = requests.get(url, params=params)
    data = r.json()

    rows = []
    for article in data.get("articles", []):
        rows.append({
            "date": article["publishedAt"].split("T")[0],
            "headline": article["title"]
        })

    return pd.DataFrame(rows)

In [326]:
def get_sentiment(text):
    try:
        result = sentiment_pipeline(text[:512])[0]
        return label_map[result["label"].lower()]
    except:
        return 0

In [327]:
news_df = fetch_recent_news(30)

news_df

,date,headline
0,2026-04-09,The MacBook Neo is the best thing to happen to...
1,2026-04-16,Microsoft planning Surface Laptop with an OLED...
2,2026-04-15,Microsoft counters the MacBook Neo with freebi...
3,2026-04-13,Microsoft is testing OpenClaw-like AI bots for...
4,2026-04-21,Microsoft Teams is trying to fix accidental ha...
...,...,...
79,2026-04-02,Sega Meganet: Online Gaming In 1990
80,2026-04-24,"This Week in Security: Annoyed Researchers, Da..."
81,2026-04-21,"Xbox Clears Up One Rumor About Project Helix, ..."
82,2026-04-02,"Meta, Google, and Amazon slash H-1B petitions ..."


In [328]:
import pandas as pd


news_df["sentiment"] = news_df["headline"].apply(get_sentiment)
news_df["date"] = pd.to_datetime(news_df["date"])

In [329]:
news_df.sort_values("date")

,date,headline,sentiment
54,2026-03-31,This App Makes Even the Sketchiest PDF or Word...,0
16,2026-03-31,Iran Threatens to Start Attacking Major US Tec...,-1
35,2026-04-01,Iran Threatens to Strike US Tech Companies in ...,-1
83,2026-04-02,"How AI could destroy — or save — humanity, acc...",0
79,2026-04-02,Sega Meganet: Online Gaming In 1990,0
...,...,...,...
5,2026-04-27,Microsoft and OpenAI’s famed AGI agreement is ...,0
74,2026-04-28,The Start of OpenAI’s Trial Against Elon Musk ...,0
75,2026-04-28,Xbox’s ‘Project Helix’ Console Should Copy the...,0
71,2026-04-28,Ex-Elder Scrolls Online Boss Opens Up About Mi...,-1


In [330]:
daily_sentiment = news_df.groupby("date").agg(
    sentiment_mean=("sentiment", "mean"),
).reset_index()

In [331]:
daily_sentiment 

,date,sentiment_mean
0,2026-03-31,-0.500000
1,2026-04-01,-1.000000
2,2026-04-02,-0.400000
3,2026-04-03,0.200000
4,2026-04-05,0.000000
5,2026-04-06,0.000000
6,2026-04-07,0.000000
7,2026-04-08,-1.000000
8,2026-04-09,-0.333333
9,2026-04-10,-0.285714


In [332]:
import datetime

def str_to_datetime(s):
  split = s.split('-')
  year, month, day = int(split[0]), int(split[1]), int(split[2])
  return datetime.datetime(year=year, month=month, day=day)

In [333]:
import yfinance as yf
import pandas as pd
import numpy as np

cols = ["Open", "High", "Low", "Close", "Volume"]

data = yf.download(ticker, period="60d")

# keep only needed columns
# data = data[cols].tail()

# flatten multiindex columns
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

# move Date index to column
data = data.reset_index()

# remove column index name like "Price"
data.columns.name = None

# drop missing values
data = data.dropna()

df = pd.merge(
    data,
    daily_sentiment,
    left_on="Date",
    right_on="date",
    how="left"
)

df["sentiment_mean"] = df["sentiment_mean"].fillna(0)

df.head()

[*********************100%***********************]  1 of 1 completed


,Date,Close,High,Low,Open,Volume,date,sentiment_mean
0,2026-02-04,413.246796,418.844006,408.308056,410.064058,45012400,NaT,0.0
1,2026-02-05,392.773529,407.370187,391.426597,406.512160,66289200,NaT,0.0
2,2026-02-06,400.226501,400.875015,392.025220,398.260987,53515300,NaT,0.0
3,2026-02-09,412.658142,413.945213,399.957120,403.928068,45480500,NaT,0.0
4,2026-02-10,412.328857,422.715155,411.760179,418.664403,44857900,NaT,0.0


In [334]:
df = df.drop("date",axis = 1)

In [335]:
label_map

{'positive': 1, 'negative': -1, 'neutral': 0}

In [336]:
df["label"] = label_maps[label]

In [337]:
df["sentiment_3d"] = df["sentiment_mean"].rolling(3).mean().fillna(0)
df["sentiment_7d"] = df["sentiment_mean"].rolling(7).mean().fillna(0)
df["sentiment_14d"] =df["sentiment_mean"].rolling(14).mean().fillna(0)

In [338]:
def compute_rsi(series, period=14):

    delta = series.diff()

    gain = (delta.where(delta > 0, 0)).rolling(period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(period).mean()

    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))

    return rsi

In [339]:
df["MA10"] = df["Close"].rolling(10).mean()
df["MA20"] = df["Close"].rolling(20).mean()

df["EMA10"] = df["Close"].ewm(span=10, adjust=False).mean()

df["RSI"] = compute_rsi(df["Close"], 14)

ema12 = df["Close"].ewm(span=12, adjust=False).mean()
ema26 = df["Close"].ewm(span=26, adjust=False).mean()

df["MACD"] = ema12 - ema26
df["MACD_Signal"] = df["MACD"].ewm(span=9, adjust=False).mean()
df["MACD_Hist"] = df["MACD"] - df["MACD_Signal"]

In [340]:
df.tail()

,Date,Close,High,Low,Open,Volume,sentiment_mean,label,sentiment_3d,sentiment_7d,sentiment_14d,MA10,MA20,EMA10,RSI,MACD,MACD_Signal,MACD_Hist
55,2026-04-24,424.619995,424.950012,415.799988,416.970001,27457400,0.000000,1.0,-0.129630,-0.234127,-0.256519,414.727002,391.972000,413.434644,74.966219,10.459020,5.530956,4.928064
56,2026-04-27,424.820007,427.109985,417.070007,422.380005,30867300,0.000000,1.0,0.037037,-0.162698,-0.256519,418.772003,395.374501,415.504710,75.443182,10.823969,6.589559,4.234410
57,2026-04-28,429.250000,429.920013,421.899994,424.570007,30438100,-0.333333,1.0,-0.111111,-0.210317,-0.208900,422.386005,398.889001,418.003854,75.998864,11.339937,7.539634,3.800303
58,2026-04-29,424.459991,426.820007,420.290009,424.579987,36777600,0.000000,1.0,-0.111111,-0.210317,-0.185091,423.710004,401.603500,419.177697,73.540980,11.232846,8.278277,2.954570
59,2026-04-30,404.001587,414.420013,403.720001,410.787506,13179399,0.000000,1.0,-0.111111,-0.103175,-0.164683,422.084161,403.335080,416.418404,63.002117,9.388924,8.500406,0.888518


In [341]:
df = df.dropna().reset_index()

In [342]:
df = df.tail(16)

In [343]:
# ohlcv_cols = ["Open", "High", "Low", "Close", "Volume"]
# sentiment_cols = ["sentiment_mean", "sentiment_3d", "sentiment_7d", "sentiment_14d"]

# raw features
ohlcv_raw = df[price_cols].values
sentiment_raw = df[sentiment_cols].values
label_raw = df[label_cols].values

# scale only OHLCV
ohlcv_scaled = feature_scaler.transform(ohlcv_raw)

# combine scaled OHLCV + raw sentiment
X_combined = np.concatenate([ohlcv_scaled, sentiment_raw,label_raw], axis=1)

# reshape for model
X_input = X_combined.reshape(1, 16, len(price_cols) + len(sentiment_cols) + len(label_cols))

E:\3rd_sem_project\stock_price_predictor_models\v1\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [344]:
pred_scaled = model.predict(X_input)

pred_price = close_scaler.inverse_transform(pred_scaled)

print("Predicted next price:", pred_price[0][0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 357ms/step
Predicted next price: 255.12946
